# **Evaluating Trained Agent**
---

In this doc file, we will talk about the evaluation implementation into the **src/eval.py** file.
It is cut into three main parts that orchestrate the evaluation process.

We will begin with the `Eval()` class, then the evaluation loop functions, and finally the main execution flow.

## 1. Import Required Libraries

In [ ]:
import argparse
import sys
import numpy as np
import os
import torch
import yaml
from pathlib import Path
from collections import deque

sys.path.insert(0, ".")
from src.env_utils import make_env, get_termination_reason
from src.logger import EpisodeLogger
from src.lunarAI import Agent
from src.policies import RandomPolicy, HeuristicPolicy

print("✓ All libraries imported successfully")

---

## 2. Configuration Setup

In [ ]:
# Load configuration file
cfg_path = "../config/eval_config.yaml"

with open(cfg_path) as f:
    cfg = yaml.safe_load(f)

print("Configuration loaded:")
for key, value in cfg.items():
    print(f"  {key}: {value}")

---

### **Eval Class**

The `Eval()` class initializes and manages the evaluation environment. Its goal is to set up all necessary components for testing a trained agent:
- Environment creation (without rendering)
- Logging system for evaluation metrics
- Model loading paths
- Evaluation directories

`__init__()` initializes the evaluation environment with all configuration parameters. It takes the config dictionary and config file path.

In [ ]:
class Eval():

    def __init__(self, cfg, cfg_path):
        self.run_name = cfg.get("run_name", Path(cfg_path).stem)
        self.algo = cfg.get("algo", "dqn")
        self.n_ep = cfg['n_episodes']
        self.seed = cfg['seed']
        base_dir = cfg.get("log_dir", "logs")
        self.run_folder = f"{base_dir}/eval/{self.run_name}"
        self.model_path = f"{base_dir}/brain/{self.run_name}_model.pth"
        self.max_time_step = cfg.get('max_time_steps', 1000)
        self.env = make_env(self.seed)
        self.state_size = self.env.observation_space.shape[0]
        self.action_size = self.env.action_space.n
        self.logger = EpisodeLogger(f"{self.run_folder}/{self.run_name}_eval.csv", run_name=self.run_name, verbose=False)

Key attributes:
- **`run_name`**: Identifier for the trained model to evaluate
- **`algo`**: Algorithm type ("random", "heuristic", or "dqn")
- **`model_path`**: Path to the trained model file
- **`state_size` / `action_size`**: Environment dimensions
- **`logger`**: Tracks evaluation episode statistics
- **`env`**: Created without rendering for faster evaluation

In [ ]:
# Initialize evaluation environment
eval_ctx = Eval(cfg, cfg_path)
print(f"✓ Evaluation environment initialized")
print(f"  State size: {eval_ctx.state_size}")
print(f"  Action size: {eval_ctx.action_size}")
print(f"  Total episodes: {eval_ctx.n_ep}")
print(f"  Algorithm: {eval_ctx.algo}")
print(f"  Model path: {eval_ctx.model_path}")

---

## 3. Agent Loading

The agent is loaded based on the algorithm type specified in the config. For DQN models, we handle both direct Agent serialization and state dictionary loading:
- **RandomPolicy**: No model needed, uses random actions
- **HeuristicPolicy**: No model needed, uses predefined heuristics
- **Agent (DQN)**: Loads the trained neural network from disk

In [ ]:
# Load agent based on algorithm
if eval_ctx.algo == "random":
    agent = RandomPolicy(eval_ctx.env.action_space, eval_ctx.seed)
    print(f"✓ Random agent initialized")
elif eval_ctx.algo == "heuristic":
    agent = HeuristicPolicy()
    print(f"✓ Heuristic agent initialized")
elif eval_ctx.algo == "dqn":
    if not os.path.exists(eval_ctx.model_path):
        print(f"✗ Model not found: {eval_ctx.model_path}")
        sys.exit(1)

    loaded = torch.load(eval_ctx.model_path, weights_only=False, map_location=torch.device('cpu'))
    if isinstance(loaded, Agent):
        agent = loaded
        print(f"✓ DQN agent loaded directly from {eval_ctx.model_path}")
    else:
        agent = Agent(eval_ctx.state_size, eval_ctx.action_size, cfg=cfg)
        try:
            agent.local_qnetwork.load_state_dict(loaded)
            agent.target_qnetwork.load_state_dict(agent.local_qnetwork.state_dict())
            print(f"✓ DQN agent weights loaded from state dictionary")
        except Exception:
            print("✗ Failed to load model")
            sys.exit(1)
else:
    print(f"✗ Unknown algo : {eval_ctx.algo}")
    sys.exit(1)

---

## 4. Evaluation Loop Functions

The evaluation process is split into two main functions:

### **time_step_loop()**

This function executes one complete evaluation episode. Unlike training, it uses **exploitation only** (epsilon = 0.0):
1. Selects best actions using the agent
2. Steps the environment
3. Accumulates rewards
4. Logs the episode when done

It takes: agent, initial state, and evaluation context.
Returns: final score, episode length, termination status, truncation status, and termination reason.

In [ ]:
def time_step_loop(agent, state, eval_ctx):
    """Execute one evaluation episode timestep loop"""
    score = 0.0
    length = 0

    for _ in range(0, eval_ctx.max_time_step):
        # Select best action (no exploration: epsilon = 0.0)
        if hasattr(agent, "get_action"):
            action = agent.get_action(state, 0.0)
        else:
            action = agent.select_action(state)

        # Step environment
        next_state, reward, terminated, truncated, info = eval_ctx.env.step(action)
        state = next_state
        score += reward
        length += 1

        if terminated or truncated:
            reason = get_termination_reason(next_state, terminated, truncated, info)
            eval_ctx.logger.log_episode(score=score, length=length, terminated=terminated, truncated=truncated, reason=reason)
            return score, length, terminated, truncated, reason

    return score, length, False, True, "sleep"

### **eval_loop()**

This is the main evaluation loop that runs across all episodes. It:
1. Iterates through episodes
2. Resets the environment with a seeded state
3. Executes the timestep loop for each episode
4. Tracks running average score over 100 episodes
5. Prints progress periodically
6. Generates final evaluation summary and plots

It takes: agent and evaluation context.

In [ ]:
def eval_loop(agent, eval_ctx):
    """Main evaluation loop across all episodes"""
    scores_100_episodes = deque(maxlen=100)

    for episode in range(1, eval_ctx.n_ep + 1):
        # Reset environment
        state, _ = eval_ctx.env.reset(seed=eval_ctx.seed + episode)
        
        # Run episode
        score, length, terminated, truncated, reason = time_step_loop(agent, state, eval_ctx)
        scores_100_episodes.append(score)

        # Print progress every 10 episodes
        if episode % 10 == 0:
            print('Episode {} Avg Score: {:.2f}'.format(episode, np.mean(scores_100_episodes)))

    # Generate evaluation summary and plots
    eval_ctx.env.close()
    eval_ctx.logger.print_summary()
    eval_ctx.logger.generate_plots()
    print(f"✓ Done. CSV/Plots → {eval_ctx.run_folder}/{eval_ctx.run_name}_eval.csv")
    
    return 0

---

## 5. Execute Evaluation

In [ ]:
# Start evaluation
print(f"\n{'='*50}")
print(f"Starting evaluation with {eval_ctx.algo} algorithm")
print(f"Run name: {eval_ctx.run_name}")
print(f"Episodes: {eval_ctx.n_ep}")
print(f"{'='*50}\n")

result = eval_loop(agent, eval_ctx)
print(f"\nEvaluation completed with result code: {result}")

---

## Summary

The evaluation pipeline consists of:

- **`Eval()`**: Manages evaluation environment, logging, and model paths
- **`time_step_loop()`**: Executes a single evaluation episode with **exploitation only** (epsilon = 0.0)
- **`eval_loop()`**: Orchestrates the complete evaluation across all episodes

The key differences from training:
- No exploration: epsilon is always 0.0 (uses best actions only)
- No learning: the agent is not updated during evaluation
- Loads pre-trained models from disk
- Supports multiple agent types (Random, Heuristic, DQN)
- Generates evaluation statistics and visualizations